In [1]:
import sys; print(sys.executable)

/Users/mkml/Documents/work/Udacity/ai-programming-foundations-project/.venv/bin/python


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("AB_NYC_2019.csv")
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [4]:
missing_rpm = df["reviews_per_month"].isna()
print(missing_rpm.sum(), "missing reviews_per_month")
print((df.loc[missing_rpm, "number_of_reviews"] == 0).all(), "-> all have zero reviews?")

10052 missing reviews_per_month
True -> all have zero reviews?


In [5]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
print(df.describe().T)
print((df["price"] == 0).sum(), "listings with price 0")
print((df["availability_365"] == 0).sum(), "listings with 0 availability")

(48895, 16)
id                                  int64
name                                  str
host_id                             int64
host_name                             str
neighbourhood_group                   str
neighbourhood                         str
latitude                          float64
longitude                         float64
room_type                             str
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                           str
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object
id                                    0
name                                 16
host_id                               0
host_name                            21
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude             

In [7]:
def handle_missing_values(df):
    """
    Handle missing values and drop columns not needed for analysis.

    - Drops 'name' (free text, not analyzed) and 'host_name'
      (personal data, not needed for the analysis).
    - Fills missing 'reviews_per_month' with 0: these rows all have
      number_of_reviews == 0, so the value is structurally missing.
    - Converts 'last_review' to datetime; never-reviewed listings stay NaT.

    Returns a new DataFrame; the input is not modified.
    """
    cleaned = df.drop(columns=["name", "host_name"]).copy()
    cleaned["reviews_per_month"] = cleaned["reviews_per_month"].fillna(0)
    cleaned["last_review"] = pd.to_datetime(cleaned["last_review"])
    return cleaned

In [8]:
df[["price", "minimum_nights"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99])

,price,minimum_nights
count,48895.000000,48895.000000
mean,152.720687,7.029962
std,240.154170,20.510550
min,0.000000,1.000000
50%,106.000000,3.000000
90%,269.000000,28.000000
95%,355.000000,30.000000
99%,799.000000,45.000000
max,10000.000000,1250.000000


In [9]:
def remove_invalid_and_extreme_values(df, price_quantile=0.99, max_min_nights=365):
    """
    Remove rows with invalid or extreme price and minimum_nights values.

    - Removes listings with price == 0 (invalid nightly price).
    - Removes listings priced above the given quantile (default 99th
      percentile), since the price distribution is strongly right-skewed.
    - Removes listings with minimum_nights above max_min_nights
      (default 365), which fall outside short-term rental use.

    Prints how many rows were removed and returns a new DataFrame.
    """
    cleaned = df[df["price"] > 0].copy()
    price_cutoff = cleaned["price"].quantile(price_quantile)
    cleaned = cleaned[
        (cleaned["price"] <= price_cutoff)
        & (cleaned["minimum_nights"] <= max_min_nights)
    ]
    print(f"Price cutoff ({price_quantile:.0%} quantile): ${price_cutoff:.0f}")
    print(f"Rows removed: {len(df) - len(cleaned)} of {len(df)}")
    return cleaned

In [10]:
df_clean = remove_invalid_and_extreme_values(handle_missing_values(df))
print(df_clean.shape)
print(df_clean.isna().sum())

Price cutoff (99% quantile): $799
Rows removed: 499 of 48895
(48396, 14)
id                                   0
host_id                              0
neighbourhood_group                  0
neighbourhood                        0
latitude                             0
longitude                            0
room_type                            0
price                                0
minimum_nights                       0
number_of_reviews                    0
last_review                       9819
reviews_per_month                    0
calculated_host_listings_count       0
availability_365                     0
dtype: int64


In [11]:
before = df[df["price"] > 0]["neighbourhood_group"].value_counts(normalize=True)
after = df_clean["neighbourhood_group"].value_counts(normalize=True)
print(pd.DataFrame({"before": before, "after": after}).round(4))

                     before   after
neighbourhood_group                
Manhattan            0.4431  0.4402
Brooklyn             0.4111  0.4130
Queens               0.1159  0.1167
Bronx                0.0223  0.0225
Staten Island        0.0076  0.0076


In [12]:
removed = df[(df["price"] > 0) & ~df["id"].isin(df_clean["id"])]
print(removed["neighbourhood_group"].value_counts(normalize=True).round(3))

neighbourhood_group
Manhattan        0.725
Brooklyn         0.223
Queens           0.037
Staten Island    0.008
Bronx            0.006
Name: proportion, dtype: float64
